# Extended Figure 4 — Source Data Export

**Extended Figure 4** provides additional temporal detail for the attribution
and trajectory analyses in Figure 5.  Panel 4B shows time-binned
differential attribution scores for five bin sizes (5, 10, 20, 25, 50 ms).
Panel 4C replicates the time-binned trajectory export across a broader set
of window definitions (including non-overlapping and cumulative windows).

## Data requirements

> **Raw neural recordings and pre-computed attribution tensors are required
> to run this notebook.**
> The `.mat`/`.pkl` files are not distributed with the repository.
> Set the `MATROOT` environment variable to the directory containing
> `Both_BigGAN_FC6_Evol_Stats.pkl`, `Both_BigGAN_FC6_Evol_Stats_expsep.pkl`,
> and the pre-computed `diff_attrib_norm_bin_tsr_*ms.pkl` tensors.
>
> ```bash
> export MATROOT=/path/to/Mat_Statistics
> ```

The exported source data files are written to `../source_data/ExtendedFig4/`.

In [ ]:
import os, sys, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from os.path import join
from scipy.stats import sem

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ".."))
from neuro_data_analysis.neural_data_lib import (
    load_neural_data,
    extract_all_evol_trajectory_psth,
    pad_psth_traj,
    get_expstr,
    extract_natref_activation_array,
    extract_evol_activation_array,
    extract_all_evol_trajectory_dyna,
)
from neuro_data_analysis.neural_data_utils import get_all_masks

In [ ]:
# Set the path to the raw neural recording data.
# Override by setting the MATROOT environment variable before launching Jupyter.
matroot = os.environ.get("MATROOT", "")

BFEStats_merge, BFEStats = load_neural_data()

# Build experiment-level metadata and standard inclusion masks
from neuro_data_analysis.neural_data_lib import pad_resp_traj
resp_col, meta_df = extract_all_evol_trajectory_dyna(BFEStats)
Amsk, Bmsk, V1msk, V4msk, ITmsk, length_msk, spc_msk, sucsmsk, \
    bsl_unstable_msk, bsl_stable_msk, validmsk = get_all_masks(meta_df)

# Directory containing pre-computed attribution tensors
attribdir = join(matroot, "Attribution_Tensors")
outdir = "../source_data/ExtendedFig4"
os.makedirs(outdir, exist_ok=True)

print(f"Valid experiments: {validmsk.sum()}")

## Extended Figure 4B: Temporal attribution

For each bin size (5, 10, 20, 25, 50 ms), load the pre-computed differential
attribution tensor and export the population-mean row as a CSV for DeePSim
and BigGAN threads.  The tensors are produced by the attribution pipeline in
`analysis/temporal_attribution.py`.

In [ ]:
bin_sizes = [5, 10, 20, 25, 50]   # ms

for bs in bin_sizes:
    for space_name in ["DeePSim", "BigGAN"]:
        tsr_path = join(attribdir, f"diff_attrib_norm_bin_tsr_{space_name}_{bs}ms.pkl")
        tsr_data = pickle.load(open(tsr_path, "rb"))
        # tsr_data: dict with keys 'attrib_tsr' (N_exps, n_eigvecs, n_bins)
        #           and 'meta_df'
        attrib_tsr = tsr_data["attrib_tsr"]   # (N, n_eigvecs, n_bins)
        attrib_meta = tsr_data["meta_df"]

        # Population mean over valid experiments
        valid_rows = attrib_meta["Expi"].isin(meta_df.loc[validmsk, "Expi"])
        pop_mean = attrib_tsr[valid_rows].mean(axis=0)   # (n_eigvecs, n_bins)

        pd.DataFrame(pop_mean).to_csv(
            join(outdir, f"FigureExt4B_{space_name}_diff_attrib_norm_bin_tsr_{bs}ms.csv"),
            index=False
        )

# Export the meta dataframe once
attrib_meta.to_csv(join(outdir, "FigureExt4_meta_df.csv"), index=False)

print(f"Extended Figure 4B source data saved for bin sizes {bin_sizes} ms.")

## Extended Figure 4C: Time-binned trajectories

Re-extract activation trajectories using a set of time windows that includes
both narrow 10 ms windows and wider cumulative windows (e.g., 0–50, 50–100,
100–150, 150–200 ms).  For each window, export population-mean and SEM
normalised trajectories for DeePSim and BigGAN threads.

In [ ]:
# Combine narrow 10 ms windows and wider cumulative windows
narrow_windows  = [(t, t + 10) for t in range(0, 200, 10)]
wide_windows = [
    (0, 20), (0, 25), (0, 50),
    (20, 40), (25, 50), (40, 60), (50, 75), (50, 100),
    (60, 80), (75, 100), (80, 100),
    (100, 120), (100, 125), (100, 150),
    (120, 140), (125, 150), (140, 160), (150, 175), (150, 200),
    (160, 180), (175, 200), (180, 200),
]
all_windows = sorted(set(narrow_windows + wide_windows))

valid_idx = np.where(validmsk.values)[0]

for (t_start, t_end) in all_windows:
    rsp_wdw = range(t_start, t_end)
    resp_col_wdw, _ = extract_all_evol_trajectory_dyna(BFEStats, rsp_wdw=rsp_wdw)
    resp_extrap_arr, extrap_mask_arr, _ = pad_resp_traj(resp_col_wdw)

    resp_valid = resp_extrap_arr[valid_idx]
    mask_valid = extrap_mask_arr[valid_idx]

    max_resp = np.nanmax(
        np.where(~mask_valid[:, :, np.newaxis], resp_valid[:, :, :2], np.nan),
        axis=1, keepdims=True
    ).clip(min=1e-8)
    resp_norm = resp_valid[:, :, :2] / max_resp
    sem_norm  = resp_valid[:, :, 2:4] / max_resp
    resp_norm[mask_valid] = np.nan
    sem_norm[mask_valid]  = np.nan

    pop_mean = np.nanmean(resp_norm, axis=0)
    pop_sem  = np.nanmean(sem_norm,  axis=0)

    tag = f"{t_start}-{t_end}"
    for thr_idx, space_name in enumerate(["DeePSim", "BigGAN"]):
        pd.DataFrame(pop_mean[:, thr_idx]).to_csv(
            join(outdir, f"FigureExtended4C_src_normresp_wdw_{tag}_{space_name}_mean.csv"), index=False)
        pd.DataFrame(pop_sem[:, thr_idx]).to_csv(
            join(outdir, f"FigureExtended4C_src_normresp_wdw_{tag}_{space_name}_sem.csv"), index=False)

print(f"Extended Figure 4C source data saved for {len(all_windows)} time windows to", outdir)